In [147]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abdulsalamramatu/tutor-eval-devet/mrbench_v3_devset.json


In [148]:
!pip install textstat
!pip install lexicalrichness

In [149]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from typing import List, Optional
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from lexicalrichness import LexicalRichness
from kaggle_secrets import UserSecretsClient
from collections import Counter
import pandas as pd
import numpy as np
import textstat
import nltk
import string
import torch
import json
import re
import os

In [150]:
secrets = UserSecretsClient()
INSTRUCT_MODEL_NAME = secrets.get_secret("MODEL_ID")
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL_NAME)
instruct_model = AutoModelForSequenceClassification.from_pretrained(
    INSTRUCT_MODEL_NAME,
    torch_dtype=torch.float16,  
    device_map="auto"           
)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [151]:
with open (r"/kaggle/input/datasets/abdulsalamramatu/tutor-eval-devet/mrbench_v3_devset.json") as f:
    data=json.load(f)

# Instructional features

In [152]:

instruct_model.eval()
def predict_batch(instruct_model,tokenizer,text1_batch,text2_batch,device,max_length,):
    enc = tokenizer(
        text1_batch,
        text2_batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = instruct_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)

    pred_ids = torch.argmax(probs, dim=-1).cpu().tolist()
    prob_vectors = probs.cpu().tolist()

    return pred_ids, prob_vectors

In [153]:
def extract_last_student_turn(conversation_history):
    pattern = r'Student:(.*?)(?=Tutor:|$)'
    student_turns = re.findall(pattern, conversation_history, re.DOTALL)

    if student_turns:
        return student_turns[-1].strip()
    return None


In [154]:
print(f"Number of labels: {instruct_model.config.num_labels}")
print(f"Model type: {instruct_model.config.model_type}")
if hasattr(instruct_model.config, 'label2id'):
    print(f"Label to ID mapping: {instruct_model.config.label2id}")
if hasattr(instruct_model.config, 'id2label'):
    print(f"ID to Label mapping: {instruct_model.config.id2label}")

Number of labels: 5
Model type: roberta
Label to ID mapping: {'Nomove': 0, 'PressAccuracy': 3, 'PressReasoning': 2, 'Uptake (Restating or revoicing)': 4, 'participation management': 1}
ID to Label mapping: {0: 'Nomove', 1: 'participation management', 2: 'PressReasoning', 3: 'PressAccuracy', 4: 'Uptake (Restating or revoicing)'}


In [155]:
def predict_single(instruct_model, tokenizer, text1, text2, device=None, max_length=512):
    if device is None:
        device = next(instruct_model.parameters()).device
    
    if isinstance(device, str):
        device = torch.device(device)

    
    pred_ids, prob_vectors = predict_batch(
        instruct_model, tokenizer, 
        [text1], [text2], 
        device, max_length
    )

    id_to_label = {
        0: 'Nomove',
        1: 'participation management', 
        2: 'PressReasoning',
        3: 'PressAccuracy',
        4: 'Uptake (Restating or revoicing)'
    }
    
    prediction = pred_ids[0]
    probabilities = prob_vectors[0]
   

    return {
        'prediction': prediction,
        'prediction_label': id_to_label[prediction], 
        'probabilities': probabilities,  
        'confidence': max(probabilities),  

        
        'prob_PressReasoning': probabilities[2],
        'prob_PressAccuracy': probabilities[3],
        'prob_Uptake': probabilities[4]
    }

# Linguistic Features

In [156]:
def preprocess_text(response):
    text = re.sub(r'\s+', ' ', response.strip().lower())
    text = re.sub(r'[^\w\s]', '', response)
    
    return response

In [157]:
def get_lexical_richness(text):

    default_metrics = {
        'mtld': None,
        'word_count': 0
    }
    

    if not text or not isinstance(text, str) or len(text.strip()) == 0:
        return default_metrics
    
    try:
        lex = LexicalRichness(text)
        word_count = lex.words
        

        if word_count < 2:
            return {        
                'mtld': None,
                'word_count': word_count

            }
        
        unique_words = lex.terms
        
     
        try:
            mtld = lex.mtld(threshold=0.72)
        except:
            mtld = None

    

        return {  
            'mtld': mtld,
             'word_count': word_count

        }
    
    except Exception as e:
        print(f"Error in lexical richness calculation: {e}")
        return default_metrics  

In [158]:
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')

politeness_tokenizer=AutoTokenizer.from_pretrained("Genius1237/xlm-roberta-large-tydip")
politeness_model=AutoModelForSequenceClassification.from_pretrained("Genius1237/xlm-roberta-large-tydip").to(device)


agency_tokenizer=AutoTokenizer.from_pretrained("EnchantedStardust/bertagent-best")
agency_model=AutoModelForSequenceClassification.from_pretrained("EnchantedStardust/bertagent-best").to(device)




Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Genius1237/xlm-roberta-large-tydip
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: EnchantedStardust/bertagent-best
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [159]:
def politeness_score(text, batch_size=8):
    politeness_scores=[]
    for i in range(0, len(text), batch_size):
        batch=text[i:i+batch_size]
        inputs=politeness_tokenizer(batch,return_tensors='pt',
                                   padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs=politeness_model(**inputs)
        if outputs.logits.shape[1]==1:
            probs=torch.sigmoid(outputs.logits).flatten().tolist()
        else:
                probs=torch.softmax(outputs.logits, dim=1)[:,1].tolist()
        politeness_scores.extend(probs)
    return politeness_scores

In [160]:
def agency_score(text, batch_size=8):
    agency_scores=[]
    for i in range(0, len(text), batch_size):
        batch=text[i:i+batch_size]
        inputs=agency_tokenizer(batch,return_tensors='pt',
                                   padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs=agency_model(**inputs)

        if outputs.logits.shape[1]==1:
            probs=torch.sigmoid(outputs.logits).flatten().tolist()
        else:
            probs=torch.softmax(outputs.logits, dim=1)[:,1].tolist()
        agency_scores.extend(probs)
    return agency_scores

# Readability Features

In [161]:
def calc_readability(text):
    flesch_score=textstat.flesch_reading_ease(text)


    return {
        'flesch_score':round(flesch_score,4),
        'flesch_ease':textstat.text_standard(text,float_output=True),

    }


# Pedagogical features

In [162]:
def pedagogical_score(responses):
    
            annotations=responses['annotation']
            annotation_map={'Yes': 2, 'To some extent': 1, 'No': 0 }

            return{

                'Mistake_Identification': annotation_map.get(annotations.get('Mistake_Identification', 'No'), 0),
                'Mistake_Location': annotation_map.get(annotations.get('Mistake_Location', 'No'), 0),
                'Actionability': annotation_map.get(annotations.get('Actionability', 'No'), 0),
                'Providing_Guidance': annotation_map.get(annotations.get('Providing_Guidance', 'No'), 0)
            }

In [163]:
from tqdm import tqdm 
def process_dataset(data):
    results = []
    
    for conv in tqdm(data, desc="Processing conversations"):
        conv_id=conv['conversation_id']
        conv_history=conv['conversation_history']
        for tutor, response_data in conv["tutor_responses"].items():
            tutor_name=tutor
            tutor_response= response_data['response']
            student_mist=extract_last_student_turn(conv_history)

            preprocessed_text=preprocess_text(tutor_response)
            talkmove_pred=predict_single(model, tokenizer, student_mist, tutor_response)
            politeness=politeness_score([tutor_response])[0]
            agency=agency_score([tutor_response])[0]
            flesch_ease=textstat.flesch_reading_ease(tutor_response)
            lexical_metrics=get_lexical_richness(preprocessed_text)
            pedagogical_metric=pedagogical_score(response_data)
            readability_features=calc_readability(tutor_response)

            results.append({
                "tutor": tutor_name,
                "conversation_id":conv_id,
                'student_mistake':student_mist,
                "tutors_response":tutor_response,
                'PressReasoning_prob': talkmove_pred['probabilities'][2],
                'PressAccuracy_prob': talkmove_pred['probabilities'][3],
                'Uptake_prob': talkmove_pred['probabilities'][4],
                "politeness_score":politeness,
                "agency_score":agency,
                'flesch_reading_ease': flesch_ease,

                **lexical_metrics,
                **readability_features,
                **pedagogical_metric
            })


    
    

    return pd.DataFrame(results)


new_math_df=process_dataset(data)

Processing conversations: 100%|██████████| 300/300 [01:57<00:00,  2.56it/s]


In [164]:
new_math_df

,tutor,conversation_id,student_mistake,tutors_response,PressReasoning_prob,PressAccuracy_prob,Uptake_prob,politeness_score,agency_score,flesch_reading_ease,mtld,word_count,flesch_score,flesch_ease,Mistake_Identification,Mistake_Location,Actionability,Providing_Guidance
0,Sonnet,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"Great, you've correctly identified the cost of...",0.000816,0.009659,0.558105,0.975984,0.577917,52.050000,28.773333,26,52.0500,16.0,2,2,2,2
1,Llama318B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,Now that we know the cost of 1 pound of meat i...,0.000762,0.005924,0.908691,0.951331,0.540315,80.097647,35.840000,32,80.0976,6.0,2,1,1,1
2,Llama31405B,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"You're close, but I notice that you calculated...",0.004723,0.757324,0.013527,0.907285,0.487226,47.376429,31.568598,39,47.3764,18.0,2,2,2,2
3,GPT4,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,"That's correct. So, if 1 pound of meat costs $...",0.001601,0.690430,0.047394,0.986556,0.510410,98.252500,30.510000,27,98.2525,2.0,2,2,2,2
4,Mistral,221-362eb11a-f190-42a6-b2a4-985fafdcfa9e,The cost of 1 pound of meat is $7.00.,It seems like you've calculated the cost as if...,0.000623,0.004658,0.633301,0.302074,0.499444,72.665000,109.760000,28,72.6650,13.0,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2471,Mistral,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,It seems there might be a misunderstanding in ...,0.000394,0.010368,0.562988,0.963009,0.461775,32.434286,22.000000,22,32.4343,14.0,2,2,2,1
2472,Phi3,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,"To solve this problem, we need to add the numb...",0.000448,0.009674,0.449219,0.657242,0.510343,69.788000,61.740000,21,69.7880,10.0,0,0,0,0
2473,Sonnet,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,That's a great start and I like how you worked...,0.000611,0.007957,0.683105,0.991297,0.586221,60.765000,76.230000,33,60.7650,9.0,2,2,2,2
2474,Expert,5910-25617a89-a4ae-47bb-8812-d6b39fa4e691,Yes I worked backwards to find out how many gu...,Okay. So Hector gave 5 less than four times as...,0.000407,0.006321,0.685059,0.129152,0.482296,71.767857,37.856000,26,71.7679,7.0,2,2,2,2


In [165]:
new_math_df.to_csv('new_math_df.csv', index=False) 